Reports should be added to the knowledge base and summarized daily, but these are setup functions for starting the vector store/summary process.

## Creating Vector Store

In [0]:
!pip install openai
!pip install dotenv

In [0]:
from __future__ import annotations

import json
import os
import time
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

from dotenv import load_dotenv
from openai import OpenAI

# -----------------------------
# CONFIG - adjust as needed
# -----------------------------
DOTENV_PATH: str | None = None
if DOTENV_PATH:
    load_dotenv(DOTENV_PATH, override=True)
else:
    load_dotenv(override=True)

API_KEY = os.getenv("OPENAI_API_KEY")
if not API_KEY:
    raise ValueError("OPENAI_API_KEY missing. Add it to .env or env vars.")

client = OpenAI(api_key=API_KEY)

KNOWLEDGE_ROOT = Path(
    "/Workspace/Users/fnisbet@redventures.com/EnergySalesOpsAutomatedReporting/knowledge"
)
TARGET_SUBFOLDERS = ["businessContext", "previousReports"]

ALLOWED_EXTS = {
    ".md",
    ".txt",
    ".pdf",
    ".docx",
    ".pptx",
    ".csv",
    ".tsv",
    ".json",
    ".yaml",
    ".yml",
    ".html",
    ".htm",
}

VECTOR_STORE_NAME = "knowledge_businessContext_previousReports"

BATCH_POLL_INTERVAL_SECONDS = 5
BATCH_POLL_MAX_POLL = 120  # ~10 minutes

# -----------------------------
# HELPERS
# -----------------------------
def iter_files_in_folders(root: Path, subfolders: List[str]) -> List[Path]:
    if not root.exists():
        raise FileNotFoundError(f"Knowledge root not found: {root}")
    files: List[Path] = []
    for sub in subfolders:
        folder = root / sub
        if not folder.exists():
            print(f"Warning: subfolder not found, skipping: {folder}")
            continue
        for p in folder.rglob("*"):
            if not p.is_file():
                continue
            if p.name.startswith("."):
                continue
            if p.suffix.lower() not in ALLOWED_EXTS:
                continue
            files.append(p)
    # de-dupe, keep stable order
    seen = set()
    deduped = []
    for p in files:
        sp = str(p.resolve())
        if sp in seen:
            continue
        seen.add(sp)
        deduped.append(p)
    return deduped


def infer_attributes_from_filename(filename: str, parent_folder: Optional[str] = None) -> Dict[str, str]:
    lower = filename.lower()
    attrs: Dict[str, str] = {}
    # doc_type heuristics
    if "onboard" in lower or "onboarding" in lower or "functional onboarding" in lower:
        attrs["doc_type"] = "onboarding"
    elif "glossary" in lower:
        attrs["doc_type"] = "glossary"
    elif "weekly" in lower or "check-in" in lower or "performance" in lower:
        attrs["doc_type"] = "weekly_report"
    elif "audit" in lower:
        attrs["doc_type"] = "audit"
    elif "journey" in lower:
        attrs["doc_type"] = "journey"
    elif "explanation" in lower or "business explanation" in lower:
        attrs["doc_type"] = "explanation"
    else:
        attrs["doc_type"] = "businessContext"

    # product_area heuristics
    if "compass" in lower:
        attrs["product_area"] = "compass"
    elif "pmax" in lower:
        attrs["product_area"] = "pmax"
    elif "vf" in lower or "audit" in lower:
        attrs["product_area"] = "compliance"
    else:
        if parent_folder:
            attrs["product_area"] = parent_folder.lower()
        else:
            attrs["product_area"] = "businessContext"
    return attrs


def _item_to_dict(item: Any) -> Dict[str, Any]:
    if item is None:
        return {}
    if isinstance(item, dict):
        return item
    if hasattr(item, "model_dump"):
        try:
            return item.model_dump()
        except Exception:
            pass
    if hasattr(item, "dict"):
        try:
            return item.dict()
        except Exception:
            pass
    if hasattr(item, "__dict__"):
        return {k: v for k, v in item.__dict__.items() if not k.startswith("_")}
    return {"repr": repr(item)}


# -----------------------------
# MAIN: upload files robustly + create batch with fallback + poll
# -----------------------------
def create_vector_store_with_attributes(
    *,
    knowledge_root: Path,
    target_subfolders: List[str],
    vector_store_name: str = VECTOR_STORE_NAME,
) -> Tuple[str, str]:
    print("Scanning files...")
    files = iter_files_in_folders(knowledge_root, target_subfolders)
    if not files:
        raise RuntimeError("No files found to upload. Check KNOWLEDGE_ROOT and TARGET_SUBFOLDERS.")

    print(f"Found {len(files)} files to upload:")
    for p in files:
        try:
            rel = p.relative_to(knowledge_root)
        except Exception:
            rel = p
        print(" -", rel)

    # Upload loop (defensive file id extraction)
    uploaded_ids: List[Optional[str]] = []
    upload_response_map: Dict[str, Any] = {}

    for p in files:
        print("Uploading:", p.name)
        try:
            with p.open("rb") as fh:
                up = client.files.create(file=fh, purpose="assistants")
        except Exception as exc:
            print(f"Upload failed for {p}: {type(exc).__name__}: {exc}")
            upload_response_map[str(p)] = {"error": repr(exc)}
            uploaded_ids.append(None)
            continue

        fid: Optional[str] = None
        # try multiple extraction strategies
        if hasattr(up, "id") and getattr(up, "id"):
            fid = getattr(up, "id")
        elif hasattr(up, "file_id") and getattr(up, "file_id"):
            fid = getattr(up, "file_id")
        if fid is None and isinstance(up, dict):
            fid = up.get("id") or up.get("file_id")
        if fid is None and hasattr(up, "model_dump"):
            try:
                dd = up.model_dump()
                if isinstance(dd, dict):
                    fid = dd.get("id") or dd.get("file_id")
            except Exception:
                pass
        if fid is None and hasattr(up, "dict"):
            try:
                dd = up.dict()
                if isinstance(dd, dict):
                    fid = dd.get("id") or dd.get("file_id")
            except Exception:
                pass
        if fid is None and hasattr(up, "__dict__"):
            dd = {k: v for k, v in up.__dict__.items() if not k.startswith("_")}
            fid = dd.get("id") or dd.get("file_id")

        # snapshot for debugging
        try:
            snapshot = up.model_dump() if hasattr(up, "model_dump") else (up if isinstance(up, dict) else repr(up))
        except Exception:
            snapshot = repr(up)
        upload_response_map[str(p)] = {"repr": repr(up), "snapshot": snapshot}

        if not fid:
            print("Warning: couldn't extract file id for uploaded file:", p)
            uploaded_ids.append(None)
        else:
            print("Uploaded OK, file_id=", fid)
            uploaded_ids.append(fid)

    # persist upload responses
    try:
        out_map = {k: v.get("snapshot") or v.get("repr") for k, v in upload_response_map.items()}
        Path("upload_responses.json").write_text(json.dumps(out_map, default=str, indent=2), encoding="utf-8")
        print("Wrote upload_responses.json for inspection.")
    except Exception as exc:
        print("Failed to write upload_responses.json:", exc)

    # Prepare per-file attribute mapping and lists
    files_param: List[Dict[str, Any]] = []
    file_ids_for_batch: List[str] = []
    fileid_to_attrs: Dict[str, Dict[str, str]] = {}

    for p, fid in zip(files, uploaded_ids):
        if not fid:
            print("Skipping batch entry for", p.name, "because file_id not extracted")
            continue
        parent_folder = p.parent.name if p.parent else None
        attrs = infer_attributes_from_filename(p.name, parent_folder=parent_folder)
        files_param.append({"file_id": fid, "attributes": attrs})
        file_ids_for_batch.append(fid)
        fileid_to_attrs[fid] = attrs
        print(f"Prepared attributes for {p.name}: {attrs}")

    if not file_ids_for_batch:
        raise RuntimeError("No files available to include in file batch (no extracted file IDs).")

    # Create or get vector store
    print("Creating vector store:", vector_store_name)
    vs = client.vector_stores.create(name=vector_store_name)
    vs_id = getattr(vs, "id", None) or _item_to_dict(vs).get("id")
    print("Vector store id:", vs_id)

    # Attempt 1: try to create a file batch with per-file attributes (files=...).
    batch = None
    batch_id: Optional[str] = None
    try:
        print("Attempting to create file batch with per-file 'files' param (attributes included)...")
        batch_kwargs: Dict[str, Any] = {"vector_store_id": vs_id, "files": files_param}
        batch = client.vector_stores.file_batches.create(**batch_kwargs)
        batch_id = getattr(batch, "id", None) or _item_to_dict(batch).get("id")
        print("Batch created (with files param). id=", batch_id)
    except TypeError as te:
        # Common when SDK doesn't accept 'files' param
        print("files param not supported by SDK (TypeError). Falling back to creating batch with file_ids only.")
        print("TypeError:", te)
    except Exception as e:
        # Other errors - log and try fallback
        print("Creating batch with files param raised, falling back to file_ids create. Error:", type(e).__name__, e)

    # Fallback: create batch with file_ids only and then update attributes per-file using files.update(...)
    if batch_id is None:
        try:
            print("Creating file batch with file_ids (no per-file attributes at creation)...")
            batch_kwargs2: Dict[str, Any] = {"vector_store_id": vs_id, "file_ids": file_ids_for_batch}
            batch = client.vector_stores.file_batches.create(**batch_kwargs2)
            batch_id = getattr(batch, "id", None) or _item_to_dict(batch).get("id")
            print("Batch created (file_ids). id=", batch_id)
        except Exception as e:
            print("Failed to create file batch with file_ids:", type(e).__name__, e)
            raise

        # After batch creation, update file attributes per-file using vector_stores.files.update(...)
        print("Updating file attributes in-place for each uploaded file (files.update)...")
        for fid, attrs in fileid_to_attrs.items():
            try:
                updated = client.vector_stores.files.update(vector_store_id=vs_id, file_id=fid, attributes=attrs)
                print(f"Updated attributes for {fid} -> {attrs}")
            except Exception as e:
                print(f"Failed to update attributes for {fid}: {type(e).__name__}: {e}")

    # Poll the batch until complete/failed/cancelled (best-effort)
    print("Polling batch status...")
    for attempt in range(BATCH_POLL_MAX_POLL):
        try:
            batch_status_obj = client.vector_stores.file_batches.retrieve(vector_store_id=vs_id, batch_id=batch_id)
            bdict = _item_to_dict(batch_status_obj)
            status = bdict.get("status")
            file_counts = bdict.get("file_counts")
            print(f"[{attempt}] status={status} file_counts={file_counts}")
            if status in ("completed", "failed", "cancelled"):
                break
        except Exception as exc:
            print("Error retrieving batch status:", type(exc).__name__, exc)
        time.sleep(BATCH_POLL_INTERVAL_SECONDS)
    else:
        print("Polling loop exhausted without reaching completed/failed/cancelled.")

    return vs_id, batch_id


# -----------------------------
# Convenience: list files and attributes after indexing
# -----------------------------
def list_vector_store_files(vector_store_id: str):
    page = client.vector_stores.files.list(vector_store_id=vector_store_id, limit=500)
    files = getattr(page, "data", []) or []
    results = []
    for f in files:
        fd = _item_to_dict(f)
        results.append(
            {
                "id": fd.get("id") or fd.get("file_id"),
                "filename": fd.get("filename"),
                "status": fd.get("status"),
                "attributes": fd.get("attributes") or {},
                "last_error": fd.get("last_error"),
            }
        )
    print("Files in vector store:")
    for r in results:
        print(json.dumps(r, ensure_ascii=False))
    return results


# -----------------------------
# Run as script
# -----------------------------
if __name__ == "__main__":
    vs_id, batch_id = create_vector_store_with_attributes(
        knowledge_root=KNOWLEDGE_ROOT,
        target_subfolders=TARGET_SUBFOLDERS,
    )
    print("Done. VECTOR_STORE_ID =", vs_id)
    print("BATCH_ID =", batch_id)

    try:
        list_vector_store_files(vs_id)
    except Exception as exc:
        print("Failed to list vector store files:", exc)
